In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import unquote
from selenium.webdriver.support.ui import Select
import pandas as pd
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import datetime 
import warnings
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")
from IPython.display import clear_output 

In [2]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
driver.maximize_window()
wait=WebDriverWait(driver, 10)
driver.get("https://g.oempartsonline.com/oem-parts/gm-rear-hydraulic-brake-hose-92265260")
#To start with the search screen

In [3]:
cols=["Sl.No","Body & Trim","Engine & Trans"]
df= pd.DataFrame(columns=cols)
count=0

In [18]:
List=["3U2Z-12259-D",
"5U2Z-12259-C"]

In [22]:
from tqdm import tqdm
for i in tqdm(range(len(List))): #len(List)):
    print(f"running {i}th element | {List[i]}")
    search_text=f"{List[i]}"
    searchbox=wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR,'[id="main_search_8"]')))
    searchbox.clear()
    sleep(1)
    searchbox.click()
    searchbox.send_keys(search_text)
    searchbox.submit()
    try:
        driver.find_element(By.CLASS_NAME,"product-fitment-button").click()
        sleep(1)
        try:
            driver.find_element(By.CSS_SELECTOR,'[class="fitment-expander"]').click()
            print("Fitment Expanded")
        except:
            pass
        years=driver.find_elements(By.CLASS_NAME,'fitment-year')
        makes=driver.find_elements(By.CLASS_NAME,'fitment-make')
        models=driver.find_elements(By.CLASS_NAME,'fitment-model')
        trims=driver.find_elements(By.CLASS_NAME,'fitment-trim')
        trans=driver.find_elements(By.CLASS_NAME,'fitment-engine')
        for year, make, model, trim,tran in zip(years[1:],makes[1:],models[1:],trims[1:],trans[1:]):
            df.loc[count,"Sl.No"]=i
            df.loc[count,"SKU"]=driver.find_element(By.CLASS_NAME,"part_number").find_element(By.CLASS_NAME,"list-value").text
            df.loc[count,"PartName"]=driver.find_element(By.XPATH,"/html/body/div[1]/div/div/div/div[2]/div/div/div[2]/div/div[1]/div/h1").text
            try:
                df.loc[count,"Positions"]=driver.find_element(By.CLASS_NAME,"positions").find_element(By.CLASS_NAME,"list-value").text
            except:
                pass
            try:
                df.loc[count,"Replaces"]=driver.find_element(By.CLASS_NAME,"product-superseded-list").find_element(By.CLASS_NAME,"list-value").text
            except:
                pass
            try:
                df.loc[count,"Sold Qty"]=int(driver.find_element(By.CLASS_NAME,"sold-in-qty").find_element(By.CLASS_NAME,"list-value").text)
            except:
                pass
            df.loc[count,"Year"]=(year.text)
            df.loc[count,"Make"]=(make.text)
            df.loc[count,"Model"]=(model.text)
            df.at[count,"Body & Trim"]=(trim.text.split(", "))
            df.at[count,"Engine & Trans"]=(tran.text.split(", "))
            count=count+1
    except:
        continue
    clear_output(True)

  0%|          | 0/2 [00:00<?, ?it/s]

running 0th element | 3U2Z-12259-D


 50%|█████     | 1/2 [00:04<00:04,  4.23s/it]

running 1th element | 5U2Z-12259-C


100%|██████████| 2/2 [00:07<00:00,  3.73s/it]


In [23]:
df

,Sl.No,Body & Trim,Engine & Trans,SKU,PartName,Sold Qty,Year,Make,Model,Replaces
0,0,"[Base, LS, LTZ]","[3.1L V6 - Gas, 3.4L V6 - Gas]",12192375,Plug Wire Set - GM (12192375),1.0,1997,Chevrolet,Lumina,NaN
1,0,[Z34],[3.4L V6 - Gas],12192375,Plug Wire Set - GM (12192375),1.0,1997,Chevrolet,Monte Carlo,NaN
2,0,"[Base, LS]","[3.1L V6 - Gas, 3.4L V6 - Gas]",12192375,Plug Wire Set - GM (12192375),1.0,1996,Chevrolet,Lumina,NaN
3,0,[Z34],[3.4L V6 - Gas],12192375,Plug Wire Set - GM (12192375),1.0,1996,Chevrolet,Monte Carlo,NaN
4,0,"[Base, SL]",[3.4L V6 - Gas],12192375,Plug Wire Set - GM (12192375),1.0,1996,Oldsmobile,Cutlass Supreme,NaN
...,...,...,...,...,...,...,...,...,...,...
965,0,"[Base, Premier]","[4.0L V6 - Flex, 4.0L V6 - Gas]",3U2Z-12259-D,Plug Wire Set - Ford (3U2Z-12259-D),1.0,2004,Mercury,Mountaineer,NaN
966,0,"[Eddie Bauer, Limited, Postal, XLS, XLT]","[4.0L V6 - Flex, 4.0L V6 - Gas]",3U2Z-12259-D,Plug Wire Set - Ford (3U2Z-12259-D),1.0,2003,Ford,Explorer,NaN
967,0,[Base],[4.0L V6 - Flex],3U2Z-12259-D,Plug Wire Set - Ford (3U2Z-12259-D),1.0,2003,Mercury,Mountaineer,NaN
968,0,"[Eddie Bauer, Limited, Postal, XLS, XLT]","[4.0L V6 - Flex, 4.0L V6 - Gas]",3U2Z-12259-D,Plug Wire Set - Ford (3U2Z-12259-D),1.0,2002,Ford,Explorer,NaN


In [24]:
OFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\Others\GM Parts Online'

with pd.ExcelWriter(OFolder+'\GMPartsGiant-CableSet_OE PartNumber.xlsx') as writer:  
    df.to_excel(writer,index=False, sheet_name='Raw')
    # df95.to_excel(writer,index=True, sheet_name='Coverage95')
    # df2.to_excel(writer,index=True, sheet_name='CoverageFull')

In [52]:
df=pd.read_excel(r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\Others\GM Parts Online\GMPartsGiant-CableSet_OE PartNumber.xlsx')

In [53]:
for i in range(len(df)):
    df["Body & Trim"][i]=df["Body & Trim"][i].split(", ")
    df["Engine & Trans"][i]=df["Engine & Trans"][i].split(", ")

In [54]:
df

,Sl.No,Body & Trim,Engine & Trans,SKU,PartName,Sold Qty,Year,Make,Model,Replaces
0,0.0,"[Base, LS, LTZ]","[3.1L V6 - Gas, 3.4L V6 - Gas]",12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN
1,0.0,[Z34],[3.4L V6 - Gas],12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Monte Carlo,NaN
2,0.0,"[Base, LS]","[3.1L V6 - Gas, 3.4L V6 - Gas]",12192375,Plug Wire Set - GM (12192375),1,1996,Chevrolet,Lumina,NaN
3,0.0,[Z34],[3.4L V6 - Gas],12192375,Plug Wire Set - GM (12192375),1,1996,Chevrolet,Monte Carlo,NaN
4,0.0,"[Base, SL]",[3.4L V6 - Gas],12192375,Plug Wire Set - GM (12192375),1,1996,Oldsmobile,Cutlass Supreme,NaN
...,...,...,...,...,...,...,...,...,...,...
982,NaN,"[Limited, SR5]",[3.4L V6 - Gas],19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1996,Toyota,4Runner,19037-62010
983,NaN,"[DLX, SR5]",[3.4L V6 - Gas],19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1996,Toyota,T100,19037-62010
984,NaN,"[Base, SR5]",[3.4L V6 - Gas],19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1996,Toyota,Tacoma,19037-62010
985,NaN,"[Base, DX, One-Ton DLX, SR5]",[3.4L V6 - Gas],19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1995,Toyota,T100,19037-62010


In [55]:
df=df.explode('Body & Trim').explode('Engine & Trans').reset_index()

In [56]:
df

,index,Sl.No,Body & Trim,Engine & Trans,SKU,PartName,Sold Qty,Year,Make,Model,Replaces
0,0,0.0,Base,3.1L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN
1,0,0.0,Base,3.4L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN
2,0,0.0,LS,3.1L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN
3,0,0.0,LS,3.4L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN
4,0,0.0,LTZ,3.1L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN
...,...,...,...,...,...,...,...,...,...,...,...
5376,985,NaN,DX,3.4L V6 - Gas,19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1995,Toyota,T100,19037-62010
5377,985,NaN,One-Ton DLX,3.4L V6 - Gas,19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1995,Toyota,T100,19037-62010
5378,985,NaN,SR5,3.4L V6 - Gas,19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1995,Toyota,T100,19037-62010
5379,986,NaN,Base,3.4L V6 - Gas,19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1995,Toyota,Tacoma,19037-62010


In [57]:
df["Engine & Trans"][1]#.split(" ")[0].replace("L","")

'3.4L V6 - Gas'

In [34]:
df["Engine & Trans"][2].split(" ")[1][0]

'V'

In [35]:
df["Engine & Trans"][2].split(" ")[1][1]

'6'

In [36]:
df["Engine & Trans"][2].split(" ")[-1]

'Gas'

In [58]:
for i in range(len(df)):
    if df["Engine & Trans"][i].count(" ")>=1:
        df.loc[i,"Litre"]=df["Engine & Trans"][i].split(" ")[0].replace("L","")
        df.loc[i,"Engine Config"]=df["Engine & Trans"][i].split(" ")[1][0]
        df.loc[i,"#Cyl"]=df["Engine & Trans"][i].split(" ")[1][1]
        df.loc[i,"Fuel"]=df["Engine & Trans"][i].split(" ")[-1]
    else:
        pass


In [59]:
df

,index,Sl.No,Body & Trim,Engine & Trans,SKU,PartName,Sold Qty,Year,Make,Model,Replaces,Litre,Engine Config,#Cyl,Fuel
0,0,0.0,Base,3.1L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN,3.1,V,6,Gas
1,0,0.0,Base,3.4L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN,3.4,V,6,Gas
2,0,0.0,LS,3.1L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN,3.1,V,6,Gas
3,0,0.0,LS,3.4L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN,3.4,V,6,Gas
4,0,0.0,LTZ,3.1L V6 - Gas,12192375,Plug Wire Set - GM (12192375),1,1997,Chevrolet,Lumina,NaN,3.1,V,6,Gas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5376,985,NaN,DX,3.4L V6 - Gas,19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1995,Toyota,T100,19037-62010,3.4,V,6,Gas
5377,985,NaN,One-Ton DLX,3.4L V6 - Gas,19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1995,Toyota,T100,19037-62010,3.4,V,6,Gas
5378,985,NaN,SR5,3.4L V6 - Gas,19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1995,Toyota,T100,19037-62010,3.4,V,6,Gas
5379,986,NaN,Base,3.4L V6 - Gas,19037-62050,Spark Plug Wire Set - Toyota (19037-62050),1,1995,Toyota,Tacoma,19037-62010,3.4,V,6,Gas


In [61]:
OFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\Others\GM Parts Online'

with pd.ExcelWriter(OFolder+'\GMPartsGiant-CableSet_OE PartNumber_expanded.xlsx') as writer:  
    df.to_excel(writer,index=False, sheet_name='Raw')
    # df95.to_excel(writer,index=True, sheet_name='Coverage95')
    # df2.to_excel(writer,index=True, sheet_name='CoverageFull')